# Multimodal Agentic Application — Part 2

Install the pinned series requirements and run all cells in order. Local CPU image/speech checkpoints download on first use. MMS-TTS English has a CC-BY-NC-4.0 license; TinySD uses CreativeML Open RAIL-M. Review their model cards before reuse. The local examples require no key. Hosted CrewAI routing and vision require your own `OPENAI_API_KEY`, `RUN_OPENAI=1` and a budget; otherwise those cells do not make requests. No Kimi, Groq, Replicate or Tavily execution is implied. [Part 1](./Part_1_Setup.ipynb) covers setup. Keep private images out of hosted requests.

In [ ]:
from pathlib import Path
import os,sys,tempfile,json
folder=Path.cwd()
if not (folder/'config.py').exists():
    folder=folder/'series'/'multimodal-agentic-workflow'
assert (folder/'config.py').is_file(), 'Open this notebook from the companion repository.'
sys.path.insert(0,str(folder))
from config import make_llm
print('Application files:', all((folder/f).is_file() for f in ['config.py','helper_tools.py','agentic_tools.py','agents.py','tasks.py','main.py']))

In [ ]:
import torch,numpy as np
from scipy.io import wavfile
from helper_tools import Helpers
torch.set_num_threads(4)
helpers=Helpers(tempfile.mkdtemp(prefix='multimodal-notebook-'))
speech=helpers.execute('text2speech','Hello, welcome to our multimodal workflow.')
rate,wave=wavfile.read(speech.artifact)
assert rate==16000 and wave.ndim==1 and np.any(wave) and np.isfinite(wave).all()
print('Speech:',rate,'Hz;',len(wave),'samples;',round(len(wave)/rate,3),'seconds')

In [ ]:
from PIL import Image
image=helpers.execute('text2image','A small blue ceramic teapot on a clean white table, studio photograph')
with Image.open(image.artifact) as im:
    pixels=np.asarray(im)
    assert im.size==(256,256) and pixels.std()>2
print('Generated image:',pixels.shape,'; nonconstant pixels:',bool(pixels.std()>2))

In [ ]:
search=helpers.execute('web_search','Python programming language official documentation')
assert search.text and search.sources and all(u.startswith('https://') for u in search.sources)
print(search.text)
print('Sources:',*search.sources,sep='\n')

In [ ]:
from agentic_tools import ExecutionState,TaskExecutorTool,Route
state=ExecutionState('original request',route=Route(tool='unsupported'))
tool=TaskExecutorTool(state,helpers)
result=tool._run()
assert state.attempted and state.error and state.result is None
try:tool._run()
except RuntimeError:pass
else:raise AssertionError('Repeated execution accepted')
assert 'failed' in result
print('Failed and repeated operations cannot become successful results.')

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from main import run_workflow
LIVE=os.getenv('RUN_OPENAI')=='1'
llm=make_llm() if LIVE else None
def run(question,image_path=''):
    # A worker thread keeps CrewAI's synchronous lifecycle outside Jupyter's event loop.
    with ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(run_workflow,question,image_path,llm=llm,helpers=helpers).result()
print('Hosted routing/vision opt-in:',LIVE)

In [ ]:
if LIVE:
    generated=run('Generate an image of a blue teapot on a white table')
    assert generated.tool=='text2image' and Path(generated.artifact).is_file()
    with Image.open(generated.artifact) as im:assert im.size==(256,256)
    print('CrewAI selected:',generated.tool,'; actual PNG size:',im.size)

In [ ]:
if LIVE:
    spoken=run('Convert this text to speech: Welcome to the multimodal workflow.')
    rate,samples=wavfile.read(spoken.artifact)
    assert spoken.tool=='text2speech' and rate==16000 and np.any(samples)
    print('CrewAI selected:',spoken.tool,'; actual WAV:',len(samples),'samples at',rate,'Hz')

In [ ]:
if LIVE:
    from PIL import ImageDraw
    chart=Path(helpers.folder)/'shapes.png'
    fixture=Image.new('RGB',(256,256),'white');draw=ImageDraw.Draw(fixture)
    draw.rectangle((30,70,105,165),fill='red');draw.ellipse((145,85,220,160),fill='blue');fixture.save(chart)
    seen=run('Describe the colors and shapes in this attached image.',str(chart))
    assert seen.tool=='image2text' and 'red' in seen.text.lower() and 'blue' in seen.text.lower()
    print(seen.text)

In [ ]:
if LIVE:
    found=run('Search the web for Python programming language official documentation')
    assert found.tool=='web_search' and found.sources and found.text
    print('CrewAI selected:',found.tool)
    print(found.text)
    print('Sources:',*found.sources,sep='\n')

In [ ]:
import shutil
assert helpers.folder.name.startswith('multimodal-notebook-')
shutil.rmtree(helpers.folder)
print('Notebook-owned generated artifacts removed.')

## Original archive appendix

Original material below is preserved for comparison, not executed.

### Original block 1

````text
from crewai.tools import BaseTool
from config import llm
from helper_tools import text2image,text2speech,image2text,web_search_tool

## Router Tool
class RouterTool(BaseTool):
    name: str = "task_router"
    description: str = "Analyzes user questions and determines which specific tool should be used. Always routes to the Task Executor agent with the tool name."

    def _run(self, question: str) -> str:
        """Router Function - Determines which tool should be used and routes to Task Executor"""
        prompt = f"""
        Based on the Question provided below, determine which tool should be used:
            1. text2image - for generating images from text descriptions
            2. text2speech - for converting text to speech
            3. image2text - for describing images based on text prompts
            4. web_search - for searching current information and answering general questions
            
            Question: {question}
            
            RESPONSE INSTRUCTIONS:
                - Answer with the exact tool name: text2image, text2speech, image2text, or web_search
                - Do not provide any preamble or explanations
                - If the question is about generating an image, return 'text2image'
                - If the question is about converting text to speech, return 'text2speech'
                - If the question is about describing an image, return 'image2text'
                - For all other questions, return 'web_search'

            OUTPUT FORMAT:
                text2image
            """
        
        # Use the LLM correctly with CrewAI
        response = llm.call(prompt)
        
        # Clean up the response to ensure it matches expected tool names
        response = response.strip().lower()
        
        if response in ['text2image', 'text2speech', 'image2text', 'web_search']:
            return response
        else:
            # Default to web_search for unknown responses
            return 'web_search'

router_tool_instance = RouterTool()
````

### Original block 2

````text
class TaskExecutorTool(BaseTool):
    name: str = "task_executor"
    description: str = "Executes the specific tool requested by the Task Router to complete user requests"

    def _run(self, router_response: str, question: str, image_url: str = None) -> str:
        """Task Executor Function - Executes the tool specified by the router"""
        try:
            if router_response == 'text2image':
                return text2image(question)
            
            elif router_response == 'text2speech':
                return text2speech(question)
            
            elif router_response == 'image2text':
                if not image_url:
                    return "Error: image_url is required for image2text operations"
                return image2text(image_url, question)
            
            elif router_response == 'web_search':
                return web_search_tool(question)
            
            else:
                return f"Error: Unknown tool '{router_response}'. Available tools: text2image, text2speech, image2text, web_search"
        except Exception as e:
            return f"Error executing {router_response}: {str(e)}"

retriver_tool_instance = TaskExecutorTool()
````

### Original block 3

````text
import replicate
from langchain_community.tools.tavily_search import TavilySearchResults

def web_search_tool(question: str) -> str:
    """This tool is useful when we want web search for current events."""
    # Function logic here
    # Step 1: Instantiate the Tavily client with your API key
    websearch = TavilySearchResults()
    # Step 2: Perform a search query
    response = websearch.invoke({"query":question})
    return response
````

### Original block 4

````text
def text2speech(text:str) -> str:
    """This tool is useful when we want to convert text to speech."""
    try:
        # Function logic here
        output = replicate.run(
        "cjwbw/seamless_communication:668a4fec05a887143e5fe8d45df25ec4c794dd43169b9a11562309b2d45873b0",
        input={
            "task_name": "T2ST (Text to Speech translation)",
            "input_text": text,
            "input_text_language": "English",
            "max_input_audio_length": 60,
            "target_language_text_only": "English",
            "target_language_with_speech": "English"
        }
        )
        return output["audio_output"]
    except Exception as e:
        if "Billing required" in str(e) or "402" in str(e):
            return f"Text-to-speech service requires billing setup. Please visit https://replicate.com/account/billing to set up billing. Original text: '{text}'"
        else:
            return f"Error in text-to-speech conversion: {str(e)}"
````

### Original block 5

````text
# Create text to image tool
def text2image(text:str) -> str:
    """This tool is useful when we want to generate images from textual descriptions."""
    try:
        # Function logic here
        output = replicate.run(
        "adirik/flux-cinestill:216a43b9975de9768114644bbf8cd0cba54a923c6d0f65adceaccfc9383a938f",
        input={
            "steps": 28,
            "prompt": text,
            "lora_url": "",
            "control_type": "depth",
            "control_image": "https://replicate.delivery/pbxt/LUSNInCegT0XwStCCJjXOojSBhPjpk2Pzj5VNjksiP9cER8A/ComfyUI_02172_.png",
            "lora_strength": 1,
            "output_format": "webp",
            "guidance_scale": 2.5,
            "output_quality": 100,
            "negative_prompt": "low quality, ugly, distorted, artefacts",
            "control_strength": 0.45,
            "depth_preprocessor": "DepthAnything",
            "soft_edge_preprocessor": "HED",
            "image_to_image_strength": 0,
            "return_preprocessed_image": False
            }
        )
        print(output)
        return output[0]
    except Exception as e:
        if "Billing required" in str(e) or "402" in str(e):
            return f"Image generation service requires billing setup. Please visit https://replicate.com/account/billing to set up billing. Requested image description: '{text}'"
        else:
            return f"Error in image generation: {str(e)}"
````

### Original block 6

````text
## text to image
def image2text(image_url:str,prompt:str) -> str:
  """This tool is useful when we want to generate textual descriptions from images."""
  try:
    # Function
    output = replicate.run(
      "adirik/flux-cinestill:216a43b9975de9768114644bbf8cd0cba54a923c6d0f65adceaccfc9383a938f",
      input={
          "image": image_url,
          "top_p": 1,
          "prompt": prompt,
          "max_tokens": 1024,
          "temperature": 0.2
      }
    )
    return "".join(output)
  except Exception as e:
    if "Billing required" in str(e) or "402" in str(e):
        return f"Image analysis service requires billing setup. Please visit https://replicate.com/account/billing to set up billing. Image URL: {image_url}, Prompt: '{prompt}'"
    else:
        return f"Error in image analysis: {str(e)}"
````

### Original block 7

````text
from crewai import Agent
from agentic_tools import router_tool_instance,retriver_tool_instance
from config import llm

# set up the router agent
Task_Router_Agent = Agent(
  role='Task Router',
  goal='Analyze user questions and determine which specific tool should be used to handle the request',
  backstory=(
    "You are an expert task router that analyzes user questions and determines the most appropriate tool to use."
    "You always route to the Task Executor agent with the specific tool name that should be used."
    "Available tools: text2image (for generating images from text), text2speech (for converting text to speech), "
    "image2text (for describing images), and web_search (for searching current information)."
    "You do not execute any tools yourself - you only determine which tool should be used and pass that information to the Task Executor."
  ),
  verbose=True,
  allow_delegation=False,
  llm=llm,
  tools=[router_tool_instance],
)
````

### Original block 8

````text
# Task Executor Agent
Task_Executor_Agent = Agent(
    role="Task Executor",
    goal="Execute the specific tool requested by the Task Router to complete the user's request",
    backstory=(
        "You are a specialized task executor that receives tool instructions from the Task Router."
        "You take the tool name provided by the router and execute the corresponding tool with the user's question."
        "You handle text2image, text2speech, image2text, and web_search operations."
        "You focus solely on executing the requested tool and providing the results without additional explanations."
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm,
    tools=[retriver_tool_instance],
)
````

### Original block 9

````text
from crewai import Task
from agents import Task_Router_Agent,Task_Executor_Agent

router_task = Task(
    description=("Analyze the user question '{question}' and determine which specific tool should be used. "
                 "Available tools and their use cases: "
                 "- text2image: for generating images from text descriptions "
                 "- text2speech: for converting text to speech "
                 "- image2text: for describing images based on text prompts "
                 "- web_search: for searching current information and answering general questions "
                 "Return the exact tool name that should be used."),
    expected_output=("A single tool name: 'text2image', 'text2speech', 'image2text', or 'web_search' based on the question analysis."),
    agent=Task_Router_Agent,
)
````

### Original block 10

````text
retriever_task = Task(
    description=("Execute the specific tool requested by the Task Router. "
                "Use the tool name provided by the router to perform the appropriate action: "
                "- text2image: Generate an image using the text2image tool with the question as prompt "
                "- text2speech: Convert the question text to speech using the text2speech tool "
                "- image2text: Describe the image using the image2text tool with the provided image URL and question "
                "- web_search: Search the web using the web_search tool with the question "
                "Question: {question}, Image URL: {image_url}"),
    expected_output=("The result of executing the specific tool requested by the Task Router."),
    agent=Task_Executor_Agent,
    context=[router_task],
)
````

### Original source artwork

![Original source figure 1](./original-assets/382e331fc502-01.png)